In [ ]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import numpy as np
from tqdm import tqdm
from FlagEmbedding import FlagReranker

In [ ]:
torch.cuda.is_available()

In [ ]:
# embedding_model = SentenceTransformer('BAAI/bge-m3', device='cpu', backend='onnx')
embedding_model = SentenceTransformer(
    r'C:\Users\ThePlayer\.cache\huggingface\hub\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181', 
    device='cpu', local_files_only=True)

reranker_v2_m3 = r'C:\Users\ThePlayer\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3\snapshots\953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
raranker_base = r"C:\Users\ThePlayer\.cache\modelscope\hub\models\BAAI\bge-reranker-base"

reranker = FlagReranker(
    reranker_v2_m3, 
    device='cuda', use_fp16=True, local_files_only=True) 


In [ ]:
reranker.device

In [ ]:
client = chromadb.PersistentClient(path="./database/chroma_1")
collection = client.get_collection(name="danbooru_tags")


In [ ]:
query_text = "standing close"

recall_output = 500 # 输出太多 reranker 会有压力
rerank_output = 20 # 输出太多 llm会有压力和干扰 

In [ ]:
query_embedding = embedding_model.encode(query_text, normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=recall_output
)

print("检索到的标签:", results['documents'])

In [ ]:
candidates = results['documents'][0]
candidates = [p.replace("_", " ") for p in candidates]

# ",".join(candidates[:20])
",".join(candidates)


In [ ]:
pairs = [[query_text, candidate] for candidate in candidates]

# 计算得分 (分数越高越相关)
scores = reranker.compute_score(pairs)

# 将得分与候选标签组合并排序
reranked_results = sorted(
    zip(candidates, scores), 
    key=lambda x: x[1], 
    reverse=True
)

# 输出前 20 个最精准的结果
for tag, score in reranked_results[:rerank_output]:
    print(f"Tag: {tag}, Score: {score:.4f}")